Cell 1: Setup, load files, helper functions

In [46]:
import re
import difflib
from collections import defaultdict
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
if not RAW.exists():
    ROOT = Path(r"C:\Users\ASUS\ipl-business-intelligence")
    RAW = ROOT / "data" / "raw"

INTERIM = ROOT / "data" / "interim"
PROCESSED = ROOT / "data" / "processed"
INTERIM.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

matches    = pd.read_csv(RAW / "matches.csv")
deliveries = pd.read_csv(RAW / "deliveries.csv")
raw_2022   = pd.read_csv(RAW / "auction_2022.csv")
raw_2023   = pd.read_csv(RAW / "auction_2023.csv")
raw_2024   = pd.read_csv(RAW / "auction_2024.csv")

TEAM_MAP = {
    "chennai super kings": "CSK", "csk": "CSK",
    "delhi capitals": "DC", "delhi daredevils": "DC", "dc": "DC",
    "gujarat titans": "GT", "gujrat titans": "GT", "gt": "GT",
    "kolkata night riders": "KKR",
    "lucknow super giants": "LSG", "lsg": "LSG",
    "mumbai indians": "MI", "mi": "MI",
    "punjab kings": "PBKS", "kings xi punjab": "PBKS", "pbks": "PBKS", "kxip": "PBKS",
    "rajasthan royals": "RR", "rr": "RR",
    "royal challengers bangalore": "RCB", "royal challengers bengaluru": "RCB", "rcb": "RCB","royal challengers banglore": "RCB",
    "sunrisers hyderabad": "SRH", "srh": "SRH",
    # defunct or renamed teams (used for matches.csv)
    "deccan chargers": "DCH", "gujarat lions": "GL", "kochi tuskers kerala": "KTK",
    "pune warriors": "PWI", "pune warriors india": "PWI",
    "rising pune supergiant": "RPS", "rising pune supergiants": "RPS",
}

ROLE_MAP = {
    "batsman": "Batter", "batter": "Batter", "bowler": "Bowler",
    "allrounder": "All-Rounder", "wicketkeeper": "Wicketkeeper",
    "wkbatter": "Wicketkeeper", "wkbatsman": "Wicketkeeper",
    "wicketkeeperbatter": "Wicketkeeper",
}

def map_team(series):
    key = (series.astype(str)
                 .str.lower()
                 .str.replace("[\u200b\u200c\u200d\ufeff]", "", regex=True)   # invisible characters
                 .str.replace(r"[^a-z]+", " ", regex=True)                     # anything else -> one space
                 .str.strip())
    return key.map(TEAM_MAP)


def map_role(series):
    key = series.astype(str).str.lower().str.replace(r"[^a-z]", "", regex=True)
    return key.map(ROLE_MAP)


def rupee_text_to_crore(series):
    """'₹ 4,40,00,000' -> 4.4"""
    digits = series.astype(str).str.replace(r"\D", "", regex=True)
    return pd.to_numeric(digits, errors="coerce") / 1e7


def norm(s):
    """Lower-case, remove dots/hyphens/punctuation, collapse spaces."""
    s = str(s).lower().replace(".", " ").replace("-", " ")
    s = re.sub(r"[^a-z\s]", "", s)
    return re.sub(r"\s+", " ", s).strip()


print("Raw folder:", RAW)
for name, df in [("matches", matches), ("deliveries", deliveries), ("2022", raw_2022),
                 ("2023", raw_2023), ("2024", raw_2024)]:
    print(f"{name:<11}", df.shape)

Raw folder: c:\Users\ASUS\ipl-business-intelligence\data\raw
matches     (1095, 20)
deliveries  (260920, 17)
2022        (204, 5)
2023        (309, 7)
2024        (675, 8)


Cell 2: Standardise the three auction files into one table

In [47]:
# ---------- 2022 ----------
a22 = pd.DataFrame({
    "season": 2022,
    "player_name": raw_2022["Player_Name"].str.strip(),
    "team": map_team(raw_2022["Teams"]),
    "role": map_role(raw_2022["Type"]),
    "is_overseas": raw_2022["Nationality"].str.strip().str.lower().map({"indian": False, "overseas": True}),
    "base_price_cr": float("nan"),
    "price_cr": rupee_text_to_crore(raw_2022["Sold_Price"]),
    "acquisition": "sold",
})

# ---------- 2023 (prices in lakhs) ----------
status_23 = raw_2023["status"].astype(str).str.strip().str.upper()
nat_23 = raw_2023["nationality"].str.strip().str.lower()
a23 = pd.DataFrame({
    "season": 2023,
    "player_name": raw_2023["name"].str.strip(),
    "team": map_team(raw_2023["franchise"]),
    "role": map_role(raw_2023["player style"]),
    "is_overseas": nat_23.ne("india").where(nat_23.notna()),
    "base_price_cr": pd.to_numeric(raw_2023["base price (in lacs)"], errors="coerce") / 100,
    "price_cr": pd.to_numeric(raw_2023["final price (in lacs)"], errors="coerce") / 100,
    "acquisition": status_23.str.lower(),
})
a23 = a23[(status_23 != "UNSOLD") & a23["price_cr"].notna()]

# ---------- 2024 (already in crores; base price in rupees) ----------
cost_col = [c for c in raw_2024.columns if "COST" in c.upper() and "CR" in c.upper()][0]
a24 = pd.DataFrame({
    "season": 2024,
    "player_name": raw_2024["Players"].str.strip(),
    "team": map_team(raw_2024["Team"]),
    "role": map_role(raw_2024["TYPE"]),
    "is_overseas": None,
    "base_price_cr": pd.to_numeric(raw_2024["Base Price"], errors="coerce") / 1e7,
    "price_cr": pd.to_numeric(raw_2024[cost_col], errors="coerce"),
    "acquisition": "sold",
})
a24 = a24[a24["price_cr"] > 0]          # drop unsold players (no price)

# 2024 has no nationality column: borrow it from 2022/2023 where the same player appears
known = pd.concat([a22, a23])[["player_name", "is_overseas"]].dropna()
lookup = {norm(n): v for n, v in zip(known["player_name"], known["is_overseas"])}
a24["is_overseas"] = a24["player_name"].map(lambda n: lookup.get(norm(n)))

auction = pd.concat([a22, a23, a24], ignore_index=True)

print("Rows kept -> 2022:", len(a22), "| 2023:", len(a23), "| 2024:", len(a24))
print("\nRows per season / acquisition type:")
print(auction.groupby(["season", "acquisition"]).size())
print("\nProblems to fix (should be 0 or small):")
print("  Unmapped team:", auction["team"].isna().sum())
print("  Unmapped role:", auction["role"].isna().sum())
print("  Missing price:", auction["price_cr"].isna().sum())
print("  Missing overseas flag:", auction["is_overseas"].isna().sum())
print("  Duplicate (season, player):", auction.duplicated(["season", "player_name"]).sum())

print("\nRaw team names that failed to map:")
for name, raw, col in [("2022", raw_2022, "Teams"), ("2023", raw_2023, "franchise"), ("2024", raw_2024, "Team")]:
    print("  ", name, list(raw.loc[map_team(raw[col]).isna(), col].dropna().unique()))

auction.to_csv(INTERIM / "auction_clean.csv", index=False)

Rows kept -> 2022: 204 | 2023: 238 | 2024: 72

Rows per season / acquisition type:
season  acquisition
2022    sold           204
2023    retained       158
        sold            80
2024    sold            72
dtype: int64

Problems to fix (should be 0 or small):
  Unmapped team: 40
  Unmapped role: 0
  Missing price: 0
  Missing overseas flag: 41
  Duplicate (season, player): 0

Raw team names that failed to map:
   2022 ['Kolkata Knight Riders']
   2023 ['KKR']
   2024 ['Unsold']


Cell 3: Sanity checks on the auction table

In [48]:
print("Price summary per season (crores):")
print(auction.groupby("season")["price_cr"].describe().round(2).to_string())

print("\nTop 3 buys per season (2022 should start with Ishan Kishan, 2024 with Mitchell Starc):")
top3 = (auction.sort_values("price_cr", ascending=False)
               .groupby("season").head(3)
               .sort_values(["season", "price_cr"], ascending=[True, False]))
print(top3[["season", "player_name", "team", "acquisition", "price_cr"]].to_string(index=False))

print("\nTotal price per team and season (crores):")
print(auction.pivot_table(index="team", columns="season", values="price_cr", aggfunc="sum").round(1).to_string())

Price summary per season (crores):
        count  mean   std  min   25%   50%   75%    max
season                                                 
2022    204.0  2.70  3.32  0.2  0.20  1.05  4.00  15.25
2023    238.0  3.68  4.49  0.2  0.21  1.50  6.25  18.50
2024     72.0  3.20  4.66  0.2  0.20  1.25  4.85  24.75

Top 3 buys per season (2022 should start with Ishan Kishan, 2024 with Mitchell Starc):
 season    player_name team acquisition  price_cr
   2022   Ishan Kishan   MI        sold     15.25
   2022  Deepak Chahar  CSK        sold     14.00
   2022   Shreyas Iyer  NaN        sold     12.25
   2023     Sam Curran PBKS        sold     18.50
   2023  Cameron Green   MI        sold     17.50
   2023       KL Rahul  LSG    retained     17.00
   2024 Mitchell Starc  KKR        sold     24.75
   2024    Pat Cummins  SRH        sold     20.50
   2024 Daryl Mitchell  CSK        sold     14.00

Total price per team and season (crores):
season  2022  2023  2024
team                    
CSK 

Cell 4: Match auction names to ball-by-ball names

In [49]:
ball_names = sorted(
    set(deliveries["batter"].dropna())
    | set(deliveries["bowler"].dropna())
    | set(deliveries["non_striker"].dropna())
)
ball_norm = {b: norm(b) for b in ball_names}

by_norm = defaultdict(list)
by_surname_initial = defaultdict(list)
by_last = defaultdict(list)
for b, n in ball_norm.items():
    toks = n.split()
    if not toks:
        continue
    by_norm[n].append(b)
    by_last[toks[-1]].append(b)
    if len(toks) >= 2:
        by_surname_initial[(" ".join(toks[1:]), toks[0][0])].append(b)
norm_keys = list(by_norm)

# After reviewing CELL 5, add fixes here and re-run this cell, e.g.
# MANUAL_OVERRIDES = {"Sai Sudharsan": "B Sai Sudharsan"}
MANUAL_OVERRIDES = {
    "Rohit Sharma": "RG Sharma",
    "Cameron Green": "C Green",
    "Varun Chakaravarthy": "CV Varun",
    "Dinesh Karthik": "KD Karthik",
    "Syed Khaleel Ahmed": "KK Ahmed",
    "Dwayne Bravo": "DJ Bravo",
    "Wanindu Hasaranga": "PWH de Silva",
    "Abhinav Sadarangani": "A Manohar",
    "Abhinav Manohar Sadarangani": "A Manohar",
    "Rinku Singh": "RK Singh",
    "Karn Sharma": "KV Sharma",
    "Shahrukh Khan": "M Shahrukh Khan",
    "Rassie Van Der Dussen": "HE van der Dussen",
    "Mujeeb Rahman": None,          # replace with the exact spelling if the finder below shows one
    # wrong matches: these players are not in the ball-by-ball data
    "Finn Allen": None,
    "Upendra Yadav": None,
    "Sonu Yadav": None,
    "K.Bhagath Varma": None,
    "Ansh Patel": None,
}


def match_name(auction_name):
    """Return (ball_name or None, method, note)."""
    if auction_name in MANUAL_OVERRIDES:
        return MANUAL_OVERRIDES[auction_name], "manual", ""
    n = norm(auction_name)
    toks = n.split()
    if not toks:
        return None, "no_match", ""

    c = by_norm.get(n, [])                                   # 1. exact
    if len(c) == 1:
        return c[0], "exact", ""

    if len(toks) >= 2:                                       # 2. surname + first initial
        c = by_surname_initial.get((" ".join(toks[1:]), toks[0][0]), [])
        if len(c) == 1:
            return c[0], "surname+initial", ""
        if len(c) > 1:
            return None, "ambiguous", " | ".join(c)

    c = by_last.get(toks[-1], [])                            # 3. unique last word
    if len(c) == 1:
        bt = ball_norm[c[0]].split()
        if toks[0] in bt[:-1] or toks[0][0] == bt[0][0]:
            return c[0], "last_name_unique", ""

    close = difflib.get_close_matches(n, norm_keys, n=1, cutoff=0.85)   # 4. fuzzy
    if close and len(by_norm[close[0]]) == 1:
        return by_norm[close[0]][0], "fuzzy", f"similar to '{close[0]}'"

    return None, "no_match", ""


players = auction[["player_name"]].drop_duplicates().reset_index(drop=True)
res = players["player_name"].apply(match_name)
players["ball_name"] = res.map(lambda t: t[0])
players["match_method"] = res.map(lambda t: t[1])
players["note"] = res.map(lambda t: t[2])

print(players["match_method"].value_counts())
print(f"\nMatched: {players['ball_name'].notna().mean():.0%} of {len(players)} distinct players")

match_method
surname+initial     203
exact                77
no_match             48
manual               19
ambiguous             8
last_name_unique      6
fuzzy                 4
Name: count, dtype: int64

Matched: 83% of 365 distinct players


Cell 5: Review the matches by hand

In [50]:
good = players[players["match_method"].isin(["exact", "surname+initial"])]
print("REVIEW 1 - spot check of confident matches (should all look right):")
print(good.sample(min(15, len(good)), random_state=1)[["player_name", "ball_name", "match_method"]].to_string(index=False))

print("\nREVIEW 2 - uncertain matches (check every row by hand):")
unsure = players[players["match_method"].isin(["last_name_unique", "fuzzy", "ambiguous"])]
print(unsure.sort_values("match_method")[["player_name", "ball_name", "match_method", "note"]].to_string(index=False))

print("\nREVIEW 3 - no match (some are players who never played, which is fine):")
print(players[players["match_method"] == "no_match"].sort_values("player_name")[["player_name"]].to_string(index=False))

REVIEW 1 - spot check of confident matches (should all look right):
      player_name       ball_name    match_method
   Jonny Bairstow     JM Bairstow surname+initial
  Rachin Ravindra      R Ravindra surname+initial
   Baba Indrajith     B Indrajith surname+initial
Prabhsimran Singh    Pankaj Singh surname+initial
    Glenn Maxwell      GJ Maxwell surname+initial
  Quinton de Kock       Q de Kock surname+initial
      Daniel Sams         DR Sams surname+initial
   Mahipal Lomror       MK Lomror surname+initial
 Dwaine Pretorius     D Pretorius surname+initial
       Will Jacks        WG Jacks surname+initial
Jason Behrendorff  JP Behrendorff surname+initial
  Shakib Al Hasan Shakib Al Hasan           exact
      Tymal Mills        TS Mills surname+initial
      Odean Smith        OF Smith surname+initial
      Sean Abbott       SA Abbott surname+initial

REVIEW 2 - uncertain matches (check every row by hand):
        player_name         ball_name     match_method                     

Cell 6: Save the matched auction table

In [51]:
auction_matched = auction.merge(
    players[["player_name", "ball_name", "match_method"]], on="player_name", how="left"
)
auction_matched.to_csv(INTERIM / "auction_matched.csv", index=False)
players.to_csv(INTERIM / "auction_name_map.csv", index=False)
print("Saved auction_matched.csv and auction_name_map.csv")

Saved auction_matched.csv and auction_name_map.csv


Cell 7: Standardise matches.csv and deliveries.csv

In [52]:
def std_team(series):
    return series.astype(str).str.strip().str.lower().map(TEAM_MAP)


matches_clean = matches.copy()
for col in ["team1", "team2", "toss_winner", "winner"]:
    new = std_team(matches[col])
    bad = matches.loc[new.isna() & matches[col].notna(), col].unique()
    print(f"{col:<12} unmapped names: {list(bad)}")
    matches_clean[col] = new

# season year from the match date (the 'season' text has formats like 2007/08)
matches_clean["date"] = pd.to_datetime(matches_clean["date"])
matches_clean["season_year"] = matches_clean["date"].dt.year

print("\nMatches per season_year:")
print(matches_clean.groupby("season_year").size().to_string())
print("\nOriginal 'season' text vs season_year:")
print(pd.crosstab(matches["season"], matches_clean["season_year"]).to_string())

deliveries_clean = deliveries.copy()
for col in ["batting_team", "bowling_team"]:
    new = std_team(deliveries[col])
    bad = deliveries.loc[new.isna() & deliveries[col].notna(), col].unique()
    print(f"{col:<13} unmapped names: {list(bad)}")
    deliveries_clean[col] = new

deliveries_clean = (deliveries_clean
    .merge(matches_clean[["id", "season_year"]], left_on="match_id", right_on="id", how="left")
    .drop(columns="id"))
print("\nDeliveries with no matching match:", deliveries_clean["season_year"].isna().sum())

matches_clean.to_csv(INTERIM / "matches_clean.csv", index=False)
deliveries_clean.to_csv(INTERIM / "deliveries_clean.csv", index=False)
print("Saved matches_clean.csv and deliveries_clean.csv")

team1        unmapped names: ['Kolkata Knight Riders']
team2        unmapped names: ['Kolkata Knight Riders']
toss_winner  unmapped names: ['Kolkata Knight Riders']
winner       unmapped names: ['Kolkata Knight Riders']

Matches per season_year:
season_year
2008    58
2009    57
2010    60
2011    73
2012    74
2013    76
2014    60
2015    59
2016    60
2017    59
2018    60
2019    60
2020    60
2021    60
2022    74
2023    74
2024    71

Original 'season' text vs season_year:
season_year  2008  2009  2010  2011  2012  2013  2014  2015  2016  2017  2018  2019  2020  2021  2022  2023  2024
season                                                                                                           
2007/08        58     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0
2009            0    57     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0
2009/10         0     0    60     0     0     0     0     0

Cell 8: Player-season performance stats

In [53]:
print("extras_type values:")
print(deliveries_clean["extras_type"].value_counts(dropna=False).to_string())
print("\ndismissal_kind values:")
print(deliveries_clean["dismissal_kind"].value_counts(dropna=False).to_string())

d = deliveries_clean.copy()
d["extras_type"] = d["extras_type"].fillna("none")
d["dismissal_kind"] = d["dismissal_kind"].fillna("none")

# matches played = any ball-by-ball appearance as batter, non-striker or bowler
appear = pd.concat([
    d[["season_year", "match_id", "batter"]].rename(columns={"batter": "player"}),
    d[["season_year", "match_id", "non_striker"]].rename(columns={"non_striker": "player"}),
    d[["season_year", "match_id", "bowler"]].rename(columns={"bowler": "player"}),
]).drop_duplicates()
matches_played = appear.groupby(["season_year", "player"]).size().rename("matches_played")

# batting
d["faced"] = (d["extras_type"] != "wides").astype(int)
d["is_four"] = (d["batsman_runs"] == 4).astype(int)
d["is_six"] = (d["batsman_runs"] == 6).astype(int)
bat = d.groupby(["season_year", "batter"]).agg(
    runs=("batsman_runs", "sum"), balls_faced=("faced", "sum"),
    fours=("is_four", "sum"), sixes=("is_six", "sum"),
)
bat.index.names = ["season_year", "player"]

outs = (d[d["player_dismissed"].notna()]
        .groupby(["season_year", "player_dismissed"]).size().rename("dismissals"))
outs.index.names = ["season_year", "player"]

# bowling (byes/leg-byes are not charged to the bowler; run-outs are not bowler wickets)
charged = d["extras_type"].isin(["wides", "noballs"])
d["conceded"] = d["batsman_runs"] + d["extra_runs"].where(charged, 0)
d["legal_bowled"] = (~charged).astype(int)
bowler_kinds = ["caught", "bowled", "lbw", "stumped", "caught and bowled", "hit wicket"]
d["bowler_wicket"] = (d["is_wicket"].eq(1) & d["dismissal_kind"].isin(bowler_kinds)).astype(int)
bowl = d.groupby(["season_year", "bowler"]).agg(
    runs_conceded=("conceded", "sum"), balls_bowled=("legal_bowled", "sum"),
    wickets=("bowler_wicket", "sum"),
)
bowl.index.names = ["season_year", "player"]

stats = pd.concat([matches_played, bat, outs, bowl], axis=1).fillna(0).reset_index()
stats["strike_rate"] = (stats["runs"] / stats["balls_faced"] * 100).where(stats["balls_faced"] > 0).round(1)
stats["economy"] = (stats["runs_conceded"] / (stats["balls_bowled"] / 6)).where(stats["balls_bowled"] > 0).round(2)
print("\nPlayer-season rows:", len(stats))

extras_type values:
extras_type
NaN        246795
wides        8380
legbyes      4001
noballs      1069
byes          673
penalty         2

dismissal_kind values:
dismissal_kind
NaN                      247970
caught                     8063
bowled                     2212
run out                    1114
lbw                         800
caught and bowled           367
stumped                     358
retired hurt                 15
hit wicket                   15
obstructing the field         3
retired out                   3

Player-season rows: 2938


Cell 9: Check the stats against known results

In [54]:
# Expect roughly: top runs 2022 Buttler 863, 2023 Gill 890, 2024 Kohli 741
#                 top wickets 2022 Chahal 27, 2023 Shami 28, 2024 Harshal Patel 24
top_runs = (stats.sort_values("runs", ascending=False).groupby("season_year").head(3)
            .sort_values(["season_year", "runs"], ascending=[True, False]))
print("Top run-scorers:")
print(top_runs[["season_year", "player", "runs"]].to_string(index=False))

top_wkts = (stats.sort_values("wickets", ascending=False).groupby("season_year").head(3)
            .sort_values(["season_year", "wickets"], ascending=[True, False]))
print("\nTop wicket-takers:")
print(top_wkts[["season_year", "player", "wickets"]].to_string(index=False))

Top run-scorers:
 season_year         player  runs
        2008       SE Marsh 616.0
        2008      G Gambhir 534.0
        2008  ST Jayasuriya 514.0
        2009      ML Hayden 572.0
        2009   AC Gilchrist 495.0
        2009 AB de Villiers 465.0
        2010   SR Tendulkar 618.0
        2010      JH Kallis 572.0
        2010       SK Raina 528.0
        2011       CH Gayle 608.0
        2011        V Kohli 557.0
        2011   SR Tendulkar 553.0
        2012       CH Gayle 733.0
        2012      G Gambhir 590.0
        2012       S Dhawan 569.0
        2013     MEK Hussey 733.0
        2013       CH Gayle 720.0
        2013        V Kohli 639.0
        2014     RV Uthappa 660.0
        2014       DR Smith 566.0
        2014     GJ Maxwell 552.0
        2015      DA Warner 562.0
        2015      AM Rahane 540.0
        2015    LMP Simmons 540.0
        2016        V Kohli 973.0
        2016      DA Warner 848.0
        2016 AB de Villiers 687.0
        2017      DA Warner 641

Cell 10: Join prices with performance and save the master table

In [55]:
master = auction_matched.merge(
    stats, left_on=["season", "ball_name"], right_on=["season_year", "player"], how="left"
)
stat_cols = ["matches_played", "runs", "balls_faced", "fours", "sixes", "dismissals",
             "runs_conceded", "balls_bowled", "wickets"]
master["played"] = master["matches_played"].fillna(0) > 0
master[stat_cols] = master[stat_cols].fillna(0)
master = master.drop(columns=["season_year", "player"])

print("Auction rows:", len(master))
print("Rows with a name match:", master["ball_name"].notna().sum())
print("Rows where the player actually played that season:", master["played"].sum())

master.to_csv(PROCESSED / "player_season_master.csv", index=False)
print("Saved:", PROCESSED / "player_season_master.csv")

master.sort_values("price_cr", ascending=False).head(10)[
    ["season", "player_name", "team", "price_cr", "matches_played", "runs", "wickets", "played"]]

Auction rows: 514
Rows with a name match: 443
Rows where the player actually played that season: 368
Saved: c:\Users\ASUS\ipl-business-intelligence\data\processed\player_season_master.csv


,season,player_name,team,price_cr,matches_played,runs,wickets,played
465,2024,Mitchell Starc,KKR,24.75,13.0,9.0,17.0,True
508,2024,Pat Cummins,SRH,20.50,16.0,136.0,18.0,True
283,2023,Sam Curran,PBKS,18.50,14.0,276.0,10.0,True
281,2023,Cameron Green,MI,17.50,16.0,452.0,6.0,True
425,2023,KL Rahul,LSG,17.00,9.0,274.0,0.0,True
280,2023,Ben Stokes,CSK,16.25,2.0,15.0,0.0,True
440,2023,Rishabh Pant,DC,16.00,0.0,0.0,0.0,False
429,2023,Ravindra Jadeja,CSK,16.00,16.0,190.0,20.0,True
423,2023,Rohit Sharma,MI,16.00,16.0,332.0,0.0,True
274,2023,Nicholas Pooran,LSG,16.00,15.0,358.0,0.0,True


In [56]:
print("=== A. 2024 team names that failed to map ===")
print(raw_2024.loc[map_team(raw_2024["Team"]).isna() & raw_2024["Team"].notna(), "Team"].value_counts())

print("\nPriced players with no team in the master table:")
print(master[master["team"].isna()][["season", "player_name", "price_cr"]].head(20).to_string(index=False))

print("\nRaw 2024 row for Starc:")
print(raw_2024[raw_2024["Players"].str.contains("Starc", na=False)].to_string(index=False))

print("\n=== B. Priced players with no stats (highest price first) ===")
no_stats = master[~master["played"]].sort_values("price_cr", ascending=False)
print(no_stats[["season", "player_name", "team", "price_cr", "ball_name", "match_method"]].head(25).to_string(index=False))

print("\n=== C. Name-match rows for Rohit Sharma and Cameron Green ===")
print(players[players["player_name"].isin(["Rohit Sharma", "Cameron Green"])].to_string(index=False))


def candidates(word):
    w = norm(word)
    return [b for b, n in ball_norm.items() if w in n.split()]


print("\nSharma:", candidates("Sharma"))
print("Green: ", candidates("Green"))

=== A. 2024 team names that failed to map ===
Team
Unsold    428
Name: count, dtype: int64

Priced players with no team in the master table:
 season         player_name  price_cr
   2022         Pat Cummins      7.25
   2022        Shreyas Iyer     12.25
   2022       Mohammad Nabi      1.00
   2022         Nitish Rana      8.00
   2022        Sam Billings      2.00
   2022         Umesh Yadav      2.00
   2022         Shivam Mavi      7.25
   2022     Sheldon Jackson      0.60
   2022      Ajinkya Rahane      1.00
   2022         Rinku Singh      0.55
   2022          Anukul Roy      0.20
   2022          Alex Hales      1.50
   2022          Rasikh Dar      0.20
   2022         Tim Southee      1.50
   2022      Baba Indrajith      0.20
   2022 Chamika Karunaratne      0.50
   2022      Abhijeet Tomar      0.40
   2022           Aman Khan      0.20
   2022        Ramesh Kumar      0.20
   2022       Pratham Singh      0.20

Raw 2024 row for Starc:
 Unnamed: 0        Players Base Pric

In [57]:
print("Rows with no team:", master["team"].isna().sum())

print("\nThe three players we fixed:")
print(master[master["player_name"].isin(["Mitchell Starc", "Cameron Green", "Rohit Sharma"])]
      [["season", "player_name", "team", "price_cr", "matches_played", "runs", "wickets"]].to_string(index=False))

print("\nPriced players still without a name match (highest price first):")
print(master[master["ball_name"].isna()].sort_values("price_cr", ascending=False)
      [["season", "player_name", "team", "price_cr", "match_method"]].head(25).to_string(index=False))

Rows with no team: 40

The three players we fixed:
 season    player_name team  price_cr  matches_played  runs  wickets
   2023  Cameron Green   MI     17.50            16.0 452.0      6.0
   2023   Rohit Sharma   MI     16.00            16.0 332.0      0.0
   2024 Mitchell Starc  KKR     24.75            13.0   9.0     17.0

Priced players still without a name match (highest price first):
 season           player_name team  price_cr match_method
   2024    Dilshan Madushanka   MI      4.60     no_match
   2024            Robin Minz   GT      3.60     no_match
   2024        Sushant Mishra   GT      2.20     no_match
   2022    Dushmanta Chameera  LSG      2.00     no_match
   2024         Mujeeb Rahman  KKR      2.00       manual
   2023         Srikar Bharat   GT      1.20     no_match
   2022        Dominic Drakes   GT      1.10     no_match
   2024          Gus Atkinson  KKR      1.00     no_match
   2022            Finn Allen  RCB      0.80       manual
   2023            Finn All

In [58]:
print("Raw 2024 team values and how they map:")
for x in raw_2024["Team"].dropna().unique():
    print(f"{x!r:<35} ->", map_team(pd.Series([x])).iloc[0])

print("\nRows still without a team:")
print(master[master["team"].isna()][["season", "player_name", "price_cr"]].to_string(index=False))

Raw 2024 team values and how they map:
'Gujrat Titans'                     -> GT
'Chennai Super Kings'               -> CSK
'Delhi Capitals'                    -> DC
'Kolkata Night Riders'              -> KKR
'Punjab Kings'                      -> PBKS
'Lucknow Super Giants'              -> LSG
'Mumbai Indians'                    -> MI
'Royal Challengers Banglore'        -> RCB
'Rajasthan Royals'                  -> RR
'Sunrisers Hyderabad'               -> SRH
'Unsold'                            -> nan

Rows still without a team:
 season         player_name  price_cr
   2022         Pat Cummins      7.25
   2022        Shreyas Iyer     12.25
   2022       Mohammad Nabi      1.00
   2022         Nitish Rana      8.00
   2022        Sam Billings      2.00
   2022         Umesh Yadav      2.00
   2022         Shivam Mavi      7.25
   2022     Sheldon Jackson      0.60
   2022      Ajinkya Rahane      1.00
   2022         Rinku Singh      0.55
   2022          Anukul Roy      0.20
   2022

In [59]:
# teams each player appeared for in each season (from ball-by-ball data)
roster = pd.concat([
    d[["season_year", "batting_team", "batter"]].rename(columns={"batting_team": "team", "batter": "player"}),
    d[["season_year", "batting_team", "non_striker"]].rename(columns={"batting_team": "team", "non_striker": "player"}),
    d[["season_year", "bowling_team", "bowler"]].rename(columns={"bowling_team": "team", "bowler": "player"}),
]).drop_duplicates()
roster_set = set(zip(roster["season_year"], roster["team"], roster["player"]))

chk = master[master["ball_name"].notna() & master["played"]].copy()
chk["on_team"] = [(s, t, p) in roster_set for s, t, p in zip(chk["season"], chk["team"], chk["ball_name"])]
bad = chk[~chk["on_team"]]

print("Matched players who played that season:", len(chk))
print("...but NOT for the team in the auction file:", len(bad), "(should be small)\n")
print(bad[["season", "player_name", "team", "ball_name", "match_method", "price_cr"]]
      .sort_values("price_cr", ascending=False).head(25).to_string(index=False))


# Finder for the names I could not guess
def candidates(word):
    w = norm(word)
    return [b for b, n in ball_norm.items() if w in n.split()]

print("\nPossible ball-by-ball names:")
for w in ["Madushanka", "Drakes", "Karunaratne", "Mishra", "Minz", "Dussen"]:
    print(f"  {w:<12}", candidates(w))

Matched players who played that season: 368
...but NOT for the team in the auction file: 4 (should be small)

 season          player_name team       ball_name    match_method  price_cr
   2024       Mitchell Starc  KKR        MA Starc surname+initial     24.75
   2024        Manish Pandey  KKR       MK Pandey surname+initial      0.50
   2024 Angkrish Raghuvanshi  KKR   A Raghuvanshi surname+initial      0.20
   2024      Ramandeep Singh  KKR Ramandeep Singh           exact      0.20

Possible ball-by-ball names:
  Madushanka   []
  Drakes       []
  Karunaratne  []
  Mishra       ['A Mishra', 'MD Mishra']
  Minz         []
  Dussen       ['HE van der Dussen']


In [60]:
def find_fragment(fragment):
    f = fragment.lower()
    return [b for b, n in ball_norm.items() if f in n]

for frag in ["shahrukh", "mujeeb", "madu", "drake", "karun"]:
    print(f"{frag:<10}", find_fragment(frag))

shahrukh   ['M Shahrukh Khan']
mujeeb     ['Mujeeb Ur Rahman']
madu       []
drake      []
karun      []
